# Reconstrucción mensual del consumo — lecturas rurales/trimestrales

Este notebook trabaja **independientemente** del notebook de exploración.

## Qué hace

- Busca los archivos `historico_2022.parquet`, `historico_2023.parquet`, etc. en `Procesado`.
- Los **copia** a una carpeta separada para trabajar sin modificar los originales.
- Une automáticamente los años que existan.
- Identifica clientes con periodicidad trimestral usando lecturas de **75 a 129 días** y recurrencia cada ~3 meses.
- Reconstruye el consumo mensual usando las **fechas reales de lectura**.
- Si faltan las fechas, usa un reparto de respaldo basado en `Días Facturados`.
- Valida que el consumo reconstruido conserve el consumo original.
- Guarda la nueva serie en una ruta independiente.

In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

from pathlib import Path
from shutil import copy2
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Modifica solamente esta ruta si cambia la ubicación del proyecto
BASE_DIR = Path(r"C:\Users\Home\Documents\Datos Ebsa")

PROCESADO_DIR = BASE_DIR / "Procesado"

# Nueva carpeta independiente
WORK_DIR = BASE_DIR / "Serie_tiempo_consumo"
FUENTE_DIR = WORK_DIR / "fuente_historicos"
SALIDA_DIR = WORK_DIR / "salidas"

for carpeta in [WORK_DIR, FUENTE_DIR, SALIDA_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Originales :", PROCESADO_DIR)
print("Trabajo    :", WORK_DIR)
print("Copias     :", FUENTE_DIR)
print("Salidas    :", SALIDA_DIR)

In [ ]:
# ============================================================
# 2. COPIAR HISTÓRICOS A LA CARPETA DE TRABAJO
# ============================================================

patron_anual = re.compile(
    r"^historico_(20\d{2})\.parquet$",
    re.IGNORECASE
)

archivos_origen = [
    ruta
    for ruta in PROCESADO_DIR.glob("historico_*.parquet")
    if patron_anual.match(ruta.name)
]

archivos_origen = sorted(
    archivos_origen,
    key=lambda p: int(
        patron_anual.match(p.name).group(1)
    )
)

if not archivos_origen:
    raise FileNotFoundError(
        "No se encontraron archivos como "
        "historico_2022.parquet dentro de:\n"
        f"{PROCESADO_DIR}"
    )

archivos_trabajo = []

for origen in archivos_origen:
    destino = FUENTE_DIR / origen.name

    if (
        not destino.exists()
        or destino.stat().st_size != origen.stat().st_size
    ):
        copy2(origen, destino)
        estado = "COPIADO"
    else:
        estado = "YA EXISTE"

    archivos_trabajo.append(destino)

    print(
        f"{estado:<10} | {origen.name:<28} "
        f"| {origen.stat().st_size / 1024**2:,.1f} MB"
    )

print(f"\nHistóricos disponibles: {len(archivos_trabajo)}")

In [ ]:
# ============================================================
# 3. CARGAR Y UNIR LOS AÑOS DISPONIBLES
# ============================================================

historicos = []

for ruta in archivos_trabajo:
    print(f"Cargando: {ruta.name}")

    temp = pd.read_parquet(
        ruta,
        engine="pyarrow"
    )

    temp["archivo_historico_origen"] = ruta.name
    historicos.append(temp)

historico = pd.concat(
    historicos,
    ignore_index=True,
    sort=False
)

del historicos

historico["periodo"] = pd.to_datetime(
    historico["periodo"],
    errors="coerce"
)

historico = (
    historico
    .dropna(subset=["NIU", "periodo"])
    .sort_values(["NIU", "periodo"])
    .reset_index(drop=True)
)

print("\nHistórico unido")
print(f"Filas      : {len(historico):,}")
print(f"NIU únicos : {historico['NIU'].nunique():,}")
print(
    f"Periodo    : "
    f"{historico['periodo'].min():%Y-%m} "
    f"→ {historico['periodo'].max():%Y-%m}"
)
print(f"Columnas   : {len(historico.columns)}")

In [ ]:
# ============================================================
# 4. VALIDAR COLUMNAS NECESARIAS
# ============================================================

COLUMNAS_NECESARIAS = [
    "NIU",
    "periodo",
    "consumo_kwh_raw",
    "dias_facturados_max",
    "fecha_lectura_anterior",
    "fecha_lectura_actual",
]

faltantes = [
    col
    for col in COLUMNAS_NECESARIAS
    if col not in historico.columns
]

if faltantes:
    raise ValueError(
        "Faltan columnas necesarias:\n"
        + "\n".join(f" • {c}" for c in faltantes)
        + "\n\nRegenera el histórico con la versión "
          "que conserva consumo y fechas."
    )

print("✓ Columnas necesarias disponibles")

print("\nColumnas de consumo:")
for col in historico.columns:
    if "consumo" in col.lower():
        print(" •", col)

In [ ]:
# ============================================================
# 5. NORMALIZAR TIPOS Y AUDITAR INTERVALOS
# ============================================================

historico["consumo_kwh_raw"] = pd.to_numeric(
    historico["consumo_kwh_raw"],
    errors="coerce"
)

historico["dias_facturados_max"] = pd.to_numeric(
    historico["dias_facturados_max"],
    errors="coerce"
)

historico["fecha_lectura_anterior"] = pd.to_datetime(
    historico["fecha_lectura_anterior"],
    errors="coerce",
    dayfirst=True
)

historico["fecha_lectura_actual"] = pd.to_datetime(
    historico["fecha_lectura_actual"],
    errors="coerce",
    dayfirst=True
)

historico["dias_entre_fechas"] = (
    historico["fecha_lectura_actual"]
    - historico["fecha_lectura_anterior"]
).dt.days

historico["diferencia_dias"] = (
    historico["dias_entre_fechas"]
    - historico["dias_facturados_max"]
)

print("Días facturados:")
display(
    historico["dias_facturados_max"].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    )
)

print("\nDiferencia entre fechas y días facturados:")
display(
    historico["diferencia_dias"].describe(
        percentiles=[.25, .50, .75, .95, .99]
    )
)

## Identificación de periodicidad trimestral

Se utilizan dos señales principales:

1. Lecturas entre **75 y 129 días**.
2. Apariciones separadas aproximadamente **3 meses**.

La clasificación es por cliente (`NIU`), no por una sola fila aislada.

In [ ]:
# ============================================================
# 6. PARÁMETROS
# ============================================================

DIAS_LARGA_MIN = 75
DIAS_LARGA_MAX = 129

PCT_LECTURAS_LARGAS_MIN = 0.60
PCT_SALTOS_3_MESES_MIN = 0.50

MIN_LECTURAS_CLIENTE = 3

historico["lectura_larga"] = (
    historico["dias_facturados_max"]
    .between(
        DIAS_LARGA_MIN,
        DIAS_LARGA_MAX
    )
)

print(
    f"Lecturas dentro de "
    f"{DIAS_LARGA_MIN}-{DIAS_LARGA_MAX} días: "
    f"{historico['lectura_larga'].sum():,}"
)

In [ ]:
# ============================================================
# 7. PERFIL DE PERIODICIDAD POR NIU
# ============================================================

perfil_dias = (
    historico
    .groupby("NIU")
    .agg(
        lecturas=("periodo", "nunique"),
        lecturas_largas=("lectura_larga", "sum"),
        mediana_dias=("dias_facturados_max", "median"),
        max_dias=("dias_facturados_max", "max"),
    )
)

perfil_dias["pct_lecturas_largas"] = (
    perfil_dias["lecturas_largas"]
    / perfil_dias["lecturas"]
)

apariciones = (
    historico[["NIU", "periodo"]]
    .drop_duplicates()
    .sort_values(["NIU", "periodo"])
    .copy()
)

apariciones["mes_num"] = (
    apariciones["periodo"].dt.year * 12
    + apariciones["periodo"].dt.month
)

apariciones["salto_meses"] = (
    apariciones
    .groupby("NIU")["mes_num"]
    .diff()
)

saltos = (
    apariciones
    .dropna(subset=["salto_meses"])
    .assign(
        salto_3_meses=lambda x:
            x["salto_meses"].eq(3)
    )
    .groupby("NIU")
    .agg(
        cantidad_saltos=("salto_meses", "count"),
        saltos_3_meses=("salto_3_meses", "sum"),
    )
)

saltos["pct_saltos_3_meses"] = (
    saltos["saltos_3_meses"]
    / saltos["cantidad_saltos"]
)

perfil_clientes = (
    perfil_dias
    .join(saltos, how="left")
    .fillna(
        {
            "cantidad_saltos": 0,
            "saltos_3_meses": 0,
            "pct_saltos_3_meses": 0,
        }
    )
)

perfil_clientes["candidato_trimestral"] = (
    (perfil_clientes["lecturas"] >= MIN_LECTURAS_CLIENTE)
    &
    (
        perfil_clientes["pct_lecturas_largas"]
        >= PCT_LECTURAS_LARGAS_MIN
    )
    &
    (
        (
            perfil_clientes["pct_saltos_3_meses"]
            >= PCT_SALTOS_3_MESES_MIN
        )
        |
        (
            perfil_clientes["lecturas_largas"] >= 3
        )
    )
)

perfil_clientes = perfil_clientes.reset_index()

display(
    perfil_clientes[
        "candidato_trimestral"
    ]
    .value_counts()
    .rename_axis("candidato_trimestral")
    .to_frame("clientes")
)

display(
    perfil_clientes[
        "candidato_trimestral"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename_axis("candidato_trimestral")
    .to_frame("porcentaje")
)

In [ ]:
# ============================================================
# 8. AÑADIR CLASIFICACIÓN AL HISTÓRICO
# ============================================================

historico = historico.merge(
    perfil_clientes[
        [
            "NIU",
            "candidato_trimestral",
            "pct_lecturas_largas",
            "pct_saltos_3_meses",
            "mediana_dias",
        ]
    ],
    on="NIU",
    how="left"
)

historico["candidato_trimestral"] = (
    historico["candidato_trimestral"]
    .fillna(False)
    .astype(bool)
)

print(
    "NIU trimestrales candidatos:",
    f"{historico.loc[historico['candidato_trimestral'], 'NIU'].nunique():,}"
)

print(
    "NIU restantes:",
    f"{historico.loc[~historico['candidato_trimestral'], 'NIU'].nunique():,}"
)

## Reconstrucción por fechas reales

Para un cliente clasificado como trimestral se reconstruyen **todas sus lecturas** usando sus fechas reales.

Ejemplo: si una lectura va del 13 de enero al 14 de abril, su consumo se reparte proporcionalmente entre enero, febrero, marzo y abril según los días cubiertos.

Los aportes de lecturas consecutivas que caen en el mismo mes se suman para formar el consumo mensual.

In [ ]:
# ============================================================
# 9. FUNCIÓN DE DESAGREGACIÓN
# ============================================================

def desagregar_lectura(
    lectura_id,
    niu,
    periodo_reporte,
    consumo,
    fecha_anterior,
    fecha_actual,
    dias_facturados,
):
    """
    Genera aportes mensuales de una lectura.

    Prioridad:
    1. Fechas reales.
    2. Respaldo por Días Facturados.
    """

    if pd.isna(consumo):
        return []

    consumo = float(consumo)
    periodo_reporte = pd.Timestamp(periodo_reporte)

    # MÉTODO 1: FECHAS REALES
    if (
        pd.notna(fecha_anterior)
        and pd.notna(fecha_actual)
    ):
        fecha_anterior = pd.Timestamp(fecha_anterior)
        fecha_actual = pd.Timestamp(fecha_actual)

        dias_totales = (
            fecha_actual
            - fecha_anterior
        ).days

        # 129 es el máximo observado en Días Facturados.
        # Se deja margen hasta 160 por diferencias de fechas.
        if 1 <= dias_totales <= 160:

            resultados = []
            cursor = fecha_anterior

            while cursor < fecha_actual:

                siguiente_mes = (
                    cursor
                    .to_period("M")
                    .to_timestamp()
                    + pd.offsets.MonthBegin(1)
                )

                fin_segmento = min(
                    siguiente_mes,
                    fecha_actual
                )

                dias_segmento = (
                    fin_segmento
                    - cursor
                ).days

                if dias_segmento > 0:
                    resultados.append(
                        {
                            "lectura_id": lectura_id,
                            "NIU": niu,
                            "periodo": (
                                cursor
                                .to_period("M")
                                .to_timestamp()
                            ),
                            "consumo_kwh_aporte": (
                                consumo
                                * dias_segmento
                                / dias_totales
                            ),
                            "dias_asignados": dias_segmento,
                            "dias_intervalo": dias_totales,
                            "consumo_lectura_original": consumo,
                            "periodo_reporte_origen": periodo_reporte,
                            "metodo_imputacion":
                                "prorrateo_dias_reales",
                        }
                    )

                cursor = fin_segmento

            return resultados

    # MÉTODO 2: RESPALDO SIN FECHAS
    if (
        pd.notna(dias_facturados)
        and dias_facturados > 0
    ):
        n_meses = int(
            np.clip(
                round(
                    float(dias_facturados)
                    / 30.44
                ),
                1,
                5
            )
        )
    else:
        n_meses = 3

    consumo_mes = consumo / n_meses
    resultados = []
    periodo_base = periodo_reporte.to_period("M")

    for desplazamiento in range(
        n_meses - 1,
        -1,
        -1
    ):
        periodo_destino = (
            periodo_base
            - desplazamiento
        ).to_timestamp()

        resultados.append(
            {
                "lectura_id": lectura_id,
                "NIU": niu,
                "periodo": periodo_destino,
                "consumo_kwh_aporte": consumo_mes,
                "dias_asignados": np.nan,
                "dias_intervalo": dias_facturados,
                "consumo_lectura_original": consumo,
                "periodo_reporte_origen": periodo_reporte,
                "metodo_imputacion":
                    "reparto_igual_sin_fechas",
            }
        )

    return resultados

In [ ]:
# ============================================================
# 10. RECONSTRUIR CLIENTES TRIMESTRALES
# ============================================================

trimestrales = (
    historico[
        historico["candidato_trimestral"]
    ]
    .copy()
    .reset_index(drop=True)
)

trimestrales["lectura_id"] = np.arange(
    len(trimestrales),
    dtype=np.int64
)

print(
    f"Lecturas de clientes trimestrales: "
    f"{len(trimestrales):,}"
)

aportes = []

columnas_lectura = [
    "lectura_id",
    "NIU",
    "periodo",
    "consumo_kwh_raw",
    "fecha_lectura_anterior",
    "fecha_lectura_actual",
    "dias_facturados_max",
]

for fila in trimestrales[
    columnas_lectura
].itertuples(
    index=False,
    name=None
):
    (
        lectura_id,
        niu,
        periodo,
        consumo,
        fecha_anterior,
        fecha_actual,
        dias_facturados,
    ) = fila

    aportes.extend(
        desagregar_lectura(
            lectura_id=lectura_id,
            niu=niu,
            periodo_reporte=periodo,
            consumo=consumo,
            fecha_anterior=fecha_anterior,
            fecha_actual=fecha_actual,
            dias_facturados=dias_facturados,
        )
    )

aportes_trimestrales = pd.DataFrame(
    aportes
)

print(
    f"Aportes mensuales generados: "
    f"{len(aportes_trimestrales):,}"
)

display(
    aportes_trimestrales.head(10)
)

In [ ]:
# ============================================================
# 11. VALIDAR CONSERVACIÓN DEL CONSUMO
# ============================================================

if aportes_trimestrales.empty:
    raise ValueError(
        "No se generaron aportes trimestrales. "
        "Revisa los umbrales de clasificación."
    )

validacion_lecturas = (
    aportes_trimestrales
    .groupby(
        "lectura_id",
        as_index=False
    )
    .agg(
        consumo_original=(
            "consumo_lectura_original",
            "first"
        ),
        consumo_reconstruido=(
            "consumo_kwh_aporte",
            "sum"
        ),
        meses_generados=(
            "periodo",
            "nunique"
        ),
        metodo=(
            "metodo_imputacion",
            "first"
        ),
    )
)

validacion_lecturas["diferencia"] = (
    validacion_lecturas[
        "consumo_reconstruido"
    ]
    - validacion_lecturas[
        "consumo_original"
    ]
)

display(
    validacion_lecturas[
        "diferencia"
    ]
    .abs()
    .describe(
        percentiles=[
            .50,
            .90,
            .95,
            .99
        ]
    )
)

print(
    "Máxima diferencia absoluta:",
    validacion_lecturas[
        "diferencia"
    ].abs().max()
)

In [ ]:
# ============================================================
# 12. CONSOLIDAR APORTES A NIU - MES
# ============================================================

serie_trimestral = (
    aportes_trimestrales
    .groupby(
        [
            "NIU",
            "periodo"
        ],
        as_index=False
    )
    .agg(
        consumo_kwh_mensual=(
            "consumo_kwh_aporte",
            "sum"
        ),
        dias_asignados=(
            "dias_asignados",
            "sum"
        ),
        lecturas_que_aportan=(
            "lectura_id",
            "nunique"
        ),
        metodos_usados=(
            "metodo_imputacion",
            lambda s:
                "|".join(
                    sorted(
                        set(
                            s.dropna()
                            .astype(str)
                        )
                    )
                )
        ),
    )
)

serie_trimestral[
    "origen_consumo"
] = "reconstruido_trimestral"

serie_trimestral[
    "consumo_imputado"
] = True

print(
    f"Filas NIU-mes reconstruidas: "
    f"{len(serie_trimestral):,}"
)

display(
    serie_trimestral.head(10)
)

In [ ]:
# ============================================================
# 13. CLIENTES NO TRIMESTRALES
# ============================================================

serie_regular = (
    historico[
        ~historico[
            "candidato_trimestral"
        ]
    ]
    .copy()
)

serie_regular = (
    serie_regular[
        [
            "NIU",
            "periodo",
            "consumo_kwh_raw"
        ]
    ]
    .rename(
        columns={
            "consumo_kwh_raw":
                "consumo_kwh_mensual"
        }
    )
)

serie_regular["dias_asignados"] = np.nan
serie_regular["lecturas_que_aportan"] = 1
serie_regular["metodos_usados"] = "observado_mes_reporte"
serie_regular["origen_consumo"] = "observado"
serie_regular["consumo_imputado"] = False

print(
    f"Filas observadas/no trimestrales: "
    f"{len(serie_regular):,}"
)

In [ ]:
# ============================================================
# 14. UNIR SERIE OBSERVADA Y RECONSTRUIDA
# ============================================================

serie_mensual = pd.concat(
    [
        serie_regular,
        serie_trimestral
    ],
    ignore_index=True,
    sort=False
)

serie_mensual = (
    serie_mensual
    .sort_values(
        [
            "NIU",
            "periodo"
        ]
    )
    .reset_index(drop=True)
)

serie_mensual = serie_mensual.merge(
    perfil_clientes[
        [
            "NIU",
            "candidato_trimestral",
            "pct_lecturas_largas",
            "pct_saltos_3_meses",
            "mediana_dias",
        ]
    ],
    on="NIU",
    how="left"
)

print("Serie mensual consolidada")
print(f"Filas      : {len(serie_mensual):,}")
print(f"NIU únicos : {serie_mensual['NIU'].nunique():,}")
print(
    f"Periodo    : "
    f"{serie_mensual['periodo'].min():%Y-%m} "
    f"→ {serie_mensual['periodo'].max():%Y-%m}"
)

display(
    serie_mensual.head(10)
)

In [ ]:
# ============================================================
# 15. LIMITAR AL RANGO DE AÑOS DISPONIBLES
# ============================================================

inicio_estudio = (
    historico["periodo"]
    .min()
    .to_period("Y")
    .start_time
)

fin_estudio = (
    historico["periodo"]
    .max()
    .to_period("Y")
    .end_time
)

serie_mensual_estudio = (
    serie_mensual[
        serie_mensual["periodo"]
        .between(
            inicio_estudio,
            fin_estudio
        )
    ]
    .copy()
)

print(
    "Rango final:",
    inicio_estudio.date(),
    "→",
    fin_estudio.date()
)

print(
    f"Filas finales: "
    f"{len(serie_mensual_estudio):,}"
)

In [ ]:
# ============================================================
# 16. AUDITORÍA MENSUAL
# ============================================================

clientes_por_mes = (
    serie_mensual_estudio
    .groupby("periodo")["NIU"]
    .nunique()
    .rename("clientes")
    .to_frame()
)

consumo_por_mes = (
    serie_mensual_estudio
    .groupby("periodo")[
        "consumo_kwh_mensual"
    ]
    .sum(min_count=1)
    .rename("consumo_kwh")
    .to_frame()
)

auditoria_mensual = (
    clientes_por_mes
    .join(
        consumo_por_mes,
        how="outer"
    )
)

display(
    auditoria_mensual
)

print("\nOrigen del consumo:")
display(
    serie_mensual_estudio[
        "origen_consumo"
    ].value_counts()
)

In [ ]:
# ============================================================
# 17. REVISAR UN CLIENTE TRIMESTRAL
# ============================================================

nius_trimestrales = (
    perfil_clientes.loc[
        perfil_clientes[
            "candidato_trimestral"
        ],
        "NIU"
    ]
)

if len(nius_trimestrales) > 0:

    NIU_EJEMPLO = (
        nius_trimestrales.iloc[0]
    )

    print(
        "NIU ejemplo:",
        NIU_EJEMPLO
    )

    print("\nLECTURAS ORIGINALES:")

    display(
        historico[
            historico["NIU"]
            == NIU_EJEMPLO
        ][
            [
                "NIU",
                "periodo",
                "consumo_kwh_raw",
                "dias_facturados_max",
                "fecha_lectura_anterior",
                "fecha_lectura_actual",
                "lectura_larga",
            ]
        ]
        .sort_values("periodo")
    )

    print(
        "\nSERIE MENSUAL RECONSTRUIDA:"
    )

    display(
        serie_mensual_estudio[
            serie_mensual_estudio["NIU"]
            == NIU_EJEMPLO
        ][
            [
                "NIU",
                "periodo",
                "consumo_kwh_mensual",
                "origen_consumo",
                "consumo_imputado",
                "lecturas_que_aportan",
                "metodos_usados",
            ]
        ]
        .sort_values("periodo")
    )

else:
    print(
        "No se encontraron candidatos "
        "trimestrales con los umbrales actuales."
    )

In [ ]:
# ============================================================
# 18. GRÁFICA DE UN CLIENTE
# ============================================================

if "NIU_EJEMPLO" in globals():

    cliente_plot = (
        serie_mensual_estudio[
            serie_mensual_estudio["NIU"]
            == NIU_EJEMPLO
        ]
        .sort_values("periodo")
    )

    plt.figure(
        figsize=(14, 5)
    )

    plt.plot(
        cliente_plot["periodo"],
        cliente_plot[
            "consumo_kwh_mensual"
        ],
        marker="o"
    )

    plt.title(
        f"Consumo mensual reconstruido "
        f"- NIU {NIU_EJEMPLO}"
    )

    plt.xlabel("Mes")
    plt.ylabel("Consumo (kWh)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 19. GUARDAR RESULTADOS
# ============================================================

ruta_serie = (
    SALIDA_DIR
    / "serie_mensual_consumo_reconstruida.parquet"
)

ruta_perfil = (
    SALIDA_DIR
    / "perfil_periodicidad_clientes.parquet"
)

ruta_validacion = (
    SALIDA_DIR
    / "validacion_conservacion_consumo.parquet"
)

ruta_aportes = (
    SALIDA_DIR
    / "aportes_lecturas_trimestrales.parquet"
)

serie_mensual_estudio.to_parquet(
    ruta_serie,
    index=False,
    engine="pyarrow"
)

perfil_clientes.to_parquet(
    ruta_perfil,
    index=False,
    engine="pyarrow"
)

validacion_lecturas.to_parquet(
    ruta_validacion,
    index=False,
    engine="pyarrow"
)

aportes_trimestrales.to_parquet(
    ruta_aportes,
    index=False,
    engine="pyarrow"
)

print("Archivos guardados:")
print(" •", ruta_serie)
print(" •", ruta_perfil)
print(" •", ruta_validacion)
print(" •", ruta_aportes)

## Nota sobre los extremos de la serie

Si solo existe `historico_2022.parquet`, la lectura de enero de 2023 todavía no está disponible. Por eso algunos consumos rurales de noviembre/diciembre de 2022 pueden quedar incompletos.

Cuando agregues `historico_2023.parquet` a `Procesado` y ejecutes nuevamente este notebook, el archivo se copiará automáticamente y podrá completar esos meses.

Lo mismo ocurre al final de 2025 si la siguiente lectura cae en enero de 2026.